## ADE 20K MSPE full training patch size 4 recipe (R2)

This notebook produces the full-training experiments for:

1. Baseline SwinUNETR (V1) -- standard MONAI patch embedding, native resolution, no MSPE.

ADE20K Dataset:
https://www.kaggle.com/datasets/awsaf49/ade20k-dataset/data


In [ ]:
!python -c "import monai" || pip install -q "monai[nibabel, tqdm]"
!python -c "import matplotlib" || pip install -q matplotlib
!python -c "import wandb" || pip install -q wandb
!python -c "import timm" || pip install -q timm
%matplotlib inline

RESUME = False # continue from disconnected run 

In [ ]:
import os
import sys

colab = True

if colab:
    from google.colab import drive, runtime, userdata
    drive.mount('/content/drive')
    dataset_dir = r"/content/drive/MyDrive/DATASETS/ADE20K/"  
    root_dir = r"/content/drive/MyDrive/RUNS/ADE20Resized/"
    mspe_code_dir = r"/content/drive/MyDrive/MSPE/"
    sys.path.append(mspe_code_dir)
    num_workers = os.cpu_count()
    batch_size = 48  # G4 instance
    cache_rate = 1.0 # G4 instance
    DEBUG_LIMIT = None
else:
    dataset_dir = r""
    root_dir = r""
    mspe_code_dir = r""
    sys.path.append(mspe_code_dir)
    num_workers = 0
    cache_rate = 0.0
    batch_size = 1    
    DEBUG_LIMIT = 1000  
    userdata = None

os.makedirs(root_dir, exist_ok=True)


In [ ]:
# Stage the ADE20K dataset
import time
if colab:
    DRIVE_ADE = "/content/drive/MyDrive/DATASETS/ADE20K"       
    DRIVE_TAR = "/content/drive/MyDrive/DATASETS/ADE20K.tar"  
    LOCAL_ADE = "/content/ADE20K"

    if not os.path.isdir(LOCAL_ADE):
        t0 = time.time()
        if not os.path.isfile(DRIVE_TAR):
            print("One-time: packing dataset into a single tar on Drive...")
            !tar -cf "{DRIVE_TAR}" -C "{os.path.dirname(DRIVE_ADE)}" "{os.path.basename(DRIVE_ADE)}"
        !tar -xf "{DRIVE_TAR}" -C /content
        print(f"dataset staged locally in {time.time() - t0:.0f} s")

    dataset_dir = LOCAL_ADE + "/"


In [ ]:
import glob
import gc
import os
import time
import random
import copy
from abc import ABC, abstractmethod
import re
from collections import Counter

import nibabel as nib
import numpy as np
import monai
import timm
import wandb
import tqdm
import matplotlib.pyplot as plt
from PIL import Image

from monai.apps import CrossValidation
from monai.config import print_config
from monai.data import CacheDataset, Dataset, create_test_image_3d, decollate_batch, DataLoader, list_data_collate, PILReader, ThreadDataLoader
from monai.data.utils import partition_dataset
from monai.handlers import (
    MeanDice,
    MLFlowHandler,
    StatsHandler,
    TensorBoardImageHandler,
    TensorBoardStatsHandler,
)
from monai.inferers import sliding_window_inference
from monai.losses import TverskyLoss, DiceLoss, DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.blocks import PatchEmbed, UnetrUpBlock
from monai.networks.layers import trunc_normal_, Conv, Norm
from monai.networks.nets import UNet
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped, MapTransform,
    RandAffined, RandFlipd, RandRotated, RandScaleIntensityd, RandShiftIntensityd, RandAdjustContrastd, RandZoomd,
    DivisiblePadd, SpatialPadd, CropForegroundd, Resized, RandSpatialCropd,
    ScaleIntensityRanged, ScaleIntensityRangePercentilesd, NormalizeIntensityd,
    AsDiscrete, AsDiscreted, CastToTyped, ToTensord,
)
from monai.utils import first, set_determinism

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR, LinearLR, SequentialLR

from swin_mspe import (
    MSPEPatchEmbedSwinNaiveRouting,
    MSPEPatchEmbedSwinOverlapping,
    MSPEPatchEmbedSwinDilatingK3,
    DEFAULT_RESOLUTIONS,
    DEFAULT_K,
    mspe_swin_forward,
    pi_resize,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bfloat16 on Ampere and later for autocast
AMP_DTYPE = torch.bfloat16 if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print(f"AMP dtype: {AMP_DTYPE}")

print_config()

In [ ]:
# Prepare dataset

def build_ade20k_split(split):
    image_dir = os.path.join(dataset_dir, "images", split)
    annotation_dir = os.path.join(dataset_dir, "annotations", split)
    images_list = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))

    dataset = []
    for image_path in tqdm.tqdm(images_list, desc=f"Indexing ADE20K {split}"):
        image_id = os.path.splitext(os.path.basename(image_path))[0]
        annotation_path = os.path.join(annotation_dir, image_id + ".png")
        dataset.append({"image": image_path, "label": annotation_path})
    return dataset

train_dataset = build_ade20k_split("training")
validation_dataset = build_ade20k_split("validation")

# Kaggle dataset copy is missing the test set
val_dataset, test_dataset = partition_dataset(validation_dataset, ratios=[1.0, 0.0])

# Local smoke test
if not colab and DEBUG_LIMIT is not None:
    train_dataset = train_dataset[:DEBUG_LIMIT]
    val_dataset = val_dataset[:16]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


In [ ]:
# Labels loading

class LoadADE20KLabeld(MapTransform):
    def __init__(self, keys="label", allow_missing_keys=False):
        super().__init__(keys, allow_missing_keys)

    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            with Image.open(d[key]) as image:
                d[key] = np.array(image, dtype=np.uint8)[None, ...]  # uint8: classes 0 ... 150
        return d


In [ ]:
# ImageNet RGB stats
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

BASE_LONGEST = 683
CROP_SIZE = 512
SCALE_JITTER = (0.5, 2.0)

RESIZE_LONGEST = 512

PATCH_SIZE = 4
STRIDE_DIVISOR = PATCH_SIZE * 16

train_transforms = Compose([
    # deterministic
    LoadImaged(
        keys=["image"],
        reader=PILReader(converter=lambda image: image.convert("RGB"), reverse_indexing=False),
    ),
    LoadADE20KLabeld(keys="label"),
    EnsureChannelFirstd(keys=["image"]),
    Resized(keys=["image", "label"], spatial_size=BASE_LONGEST, size_mode="longest", mode=("bilinear", "nearest")),
    CastToTyped(keys=["image", "label"], dtype=(torch.uint8, torch.uint8)),

    # random 
    RandZoomd(keys=["image", "label"], prob=1.0, min_zoom=SCALE_JITTER[0], max_zoom=SCALE_JITTER[1],
              mode=("bilinear", "nearest"), keep_size=False),
    # NOTE:  pad befor ScaleIntensityRanged, worth fixing for new runs
    SpatialPadd(keys=["image", "label"], spatial_size=(CROP_SIZE, CROP_SIZE), mode="constant"),  
    RandSpatialCropd(keys=["image", "label"], roi_size=(CROP_SIZE, CROP_SIZE), random_size=False),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),  # horz. flip only 

    # photometric augs
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    RandScaleIntensityd(keys=["image"], factors=0.5, prob=0.5),                    
    RandShiftIntensityd(keys=["image"], offsets=0.125, prob=0.5),                   
    RandAdjustContrastd(keys=["image"], gamma=(0.7, 1.5), prob=0.5),
    RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.5, channel_wise=True),  
    NormalizeIntensityd(keys=["image"], subtrahend=IMAGENET_MEAN, divisor=IMAGENET_STD, channel_wise=True),
    CastToTyped(keys=["image", "label"], dtype=(torch.float32, torch.int64)),
])

val_transforms = Compose([
    LoadImaged(
        keys=["image"],
        reader=PILReader(converter=lambda image: image.convert("RGB"), reverse_indexing=False),
    ),
    LoadADE20KLabeld(keys="label"),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    NormalizeIntensityd(keys=["image"], subtrahend=IMAGENET_MEAN, divisor=IMAGENET_STD, channel_wise=True),
    EnsureTyped(keys=["image", "label"]),
    CastToTyped(keys=["image", "label"], dtype=(torch.float32, torch.int64)),

    Resized(keys=["image", "label"], spatial_size=RESIZE_LONGEST, size_mode="longest", mode=("bilinear", "nearest")),
    DivisiblePadd(keys=["image", "label"], k=STRIDE_DIVISOR, mode="constant"),
])


val_transforms_fullres = Compose([
    LoadImaged(
        keys=["image"],
        reader=PILReader(converter=lambda image: image.convert("RGB"), reverse_indexing=False),
    ),
    LoadADE20KLabeld(keys="label"),
    EnsureChannelFirstd(keys=["image"]),
    ScaleIntensityRanged(keys=["image"], a_min=0.0, a_max=255.0, b_min=0.0, b_max=1.0, clip=True),
    NormalizeIntensityd(keys=["image"], subtrahend=IMAGENET_MEAN, divisor=IMAGENET_STD, channel_wise=True),
    EnsureTyped(keys=["image", "label"]),
    CastToTyped(keys=["image", "label"], dtype=(torch.float32, torch.int64)),
])

loader_kwargs = dict(num_workers=num_workers, pin_memory=False)
if num_workers > 0:
    loader_kwargs["prefetch_factor"] = 4

train_ds = CacheDataset(data=train_dataset, transform=train_transforms,
                        cache_rate=cache_rate, num_workers=num_workers)
train_loader = ThreadDataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                collate_fn=list_data_collate, **loader_kwargs)
# NOTE: MSPE Eval
val_ds = CacheDataset(data=val_dataset, transform=val_transforms,
                      cache_rate=cache_rate, num_workers=num_workers)
val_loader = ThreadDataLoader(val_ds, batch_size=1, shuffle=False, **loader_kwargs)

# NOTE: The Real eval
val_ds_fullres = CacheDataset(data=val_dataset, transform=val_transforms_fullres,
                              cache_rate=0.0, num_workers=num_workers)
val_loader_fullres = ThreadDataLoader(val_ds_fullres, batch_size=1, shuffle=False, **loader_kwargs)


In [ ]:
def _denormalize_for_display(image_chw):
    # Undo ImageNet normalization so previe look natural 
    mean = torch.tensor(IMAGENET_MEAN).view(-1, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(-1, 1, 1)
    return (image_chw * std + mean).clamp(0, 1)


def display_sample_pairs(data_loader, n_samples=3, mask_alpha=0.5):
    fig, axes = plt.subplots(n_samples, 3, figsize=(18, 4 * n_samples))
    if n_samples == 1:
        axes = [axes]

    for idx, batch in enumerate(data_loader):
        if idx >= n_samples:
            break

        image = _denormalize_for_display(batch["image"][0].cpu()).permute(1, 2, 0)
        mask = batch["label"][0][0].cpu()
        filename = os.path.basename(batch["image"].meta["filename_or_obj"][0])
        n_classes = len(torch.unique(mask))

        ax_row = axes[idx]
        ax_row[0].imshow(image)
        ax_row[0].set_title(f"Image\n{filename}", fontsize=9)
        ax_row[0].axis("off")

        ax_row[1].imshow(mask, cmap="tab20", interpolation="nearest")
        ax_row[1].set_title(f"ADE20K mask\n{n_classes} classes in image", fontsize=9)
        ax_row[1].axis("off")

        ax_row[2].imshow(image)
        ax_row[2].imshow(mask, cmap="tab20", alpha=mask_alpha, interpolation="nearest")
        ax_row[2].set_title(f"Image + mask overlay\n{filename}", fontsize=9)
        ax_row[2].axis("off")

    plt.tight_layout()
    plt.show()

# Take a look at train data 
display_sample_pairs(train_loader, n_samples=3)

In [ ]:
# Experiment configuration

# Swin-T backbone -- swin_tiny_patch4_window7_224
FEATURE_SIZE = 96
SWIN_DEPTHS = (2, 2, 6, 2)
SWIN_NUM_HEADS = (3, 6, 12, 24)

RECIPE_TAG = "R2"

MAX_EPOCHS = 150          
VAL_INTERVAL = 2
EARLY_STOPPING_PATIENCE = 30

LR = 9e-4                 # decoder LR 
BACKBONE_LR = 3e-4        # swinViT LR 
WARMUP_EPOCHS = 3
WD = 0.01                 
GRAD_CLIP = 1.0
DROPOUT_PATH_RATE = 0.3   # stochastic depth
ADE20K_NUM_CLASSES = 151
IGNORE_INDEX = 0 # ADE20K class 0

DATASET_TAG = "ADE20K"
WANDB_PROJECT = "ADE20K_MSPE_SWIN_ALL"
# ADE20K resolutions
ADE_RESOLUTIONS = [128, 256, 512]
TRAIN_RESOLUTIONS = list(ADE_RESOLUTIONS)
TEST_RESOLUTIONS = [128, 256, 512, 640]

# Swin encoder init
SWIN_INIT = "imagenet"
IMAGENET_SWIN_MODEL = "swin_tiny_patch4_window7_224.ms_in1k"

# local smoke config
if not colab:
    MAX_EPOCHS = 2
    VAL_INTERVAL = 1

# All train versions (use_v2=False is set for transfer)
EXPERIMENTS = [
    {
        "name": "BASELINE_SWIN_V1_P4",
        "description": "SwinUNETR V1 baseline, patch size 4, native-resolution training, recipe R2",
        "patch_embed_class": None,
        "use_v2": False,
        "train_mode": "native",
    },
    # {
    #     "name": "SWIN_MIXED_V1_P4",
    #     "description": "SwinUNETR V1, patch size 4, mixed-resolution training",  # For ablation
    #     "patch_embed_class": None,
    #     "use_v2": False,
    #     "train_mode": "resize_aug",
    # },
    # {
    #     "name": "ROUTING_SWIN_P4",
    #     "description": "MSPE naive routing, patch size 4, full-model training",
    #     "patch_embed_class": MSPEPatchEmbedSwinNaiveRouting,
    #     "use_v2": False,
    #     "train_mode": "mspe",
    # },
    # {
    #     "name": "OVERLAPPING_SWIN_P4",
    #     "description": "MSPE overlapping kernels, patch size 4, full-model training",
    #     "patch_embed_class": MSPEPatchEmbedSwinOverlapping,
    #     "use_v2": False,
    #     "train_mode": "mspe",
    # },
    # {
    #     "name": "DIALATING_SWIN_P4",
    #     "description": "MSPE dilating kernels, patch size 4, full-model training",
    #     "patch_embed_class": MSPEPatchEmbedSwinDilatingK3,
    #     "use_v2": False,
    #     "train_mode": "mspe",
    # },
]


# Save checkpoints in the directory this notebook lives in
if colab:
    CKPT_DIR = r"/content/drive/MyDrive/MSPE/NOTEBOOKS/ADE20K/FULL_TRAINING/"
else:
    CKPT_DIR = os.getcwd()
os.makedirs(CKPT_DIR, exist_ok=True)
EVAL_DIR = root_dir

# R2
PRETRAINED_CKPT = None

# Run names and checkpoints
for exp in EXPERIMENTS:
    exp["run_name"] = f"{exp['name']}_{RECIPE_TAG}"
    exp["checkpoint_path"] = os.path.join(CKPT_DIR, f"best_metric_{exp['name']}_{RECIPE_TAG}.pth")

print(f"Experiments to run: {[e['run_name'] for e in EXPERIMENTS]}")
print(f"Backbone: feature_size={FEATURE_SIZE}, depths={SWIN_DEPTHS}, num_heads={SWIN_NUM_HEADS}")
print(f"Patch size: {PATCH_SIZE} (deepest stride / input divisor: {STRIDE_DIVISOR})")
print(f"Swin init: {SWIN_INIT}" + (f" ({IMAGENET_SWIN_MODEL})" if SWIN_INIT == 'imagenet' else ""))
print(f"Test resolutions: {TEST_RESOLUTIONS}")
print(f"Checkpoint dir: {CKPT_DIR}")
print(f"Eval dir: {EVAL_DIR}")
print(f"Pretrained checkpoint: {PRETRAINED_CKPT}")
print(f"\nEpochs: {MAX_EPOCHS}, optimizer: AdamW, lr(head): {LR}, lr(backbone): {BACKBONE_LR}, wd: {WD}, clip: {GRAD_CLIP}, drop_path: {DROPOUT_PATH_RATE}")
print(f"Train: base longest {BASE_LONGEST}, jitter {SCALE_JITTER}, crop {CROP_SIZE}x{CROP_SIZE}, batch: {batch_size}, cache_rate: {cache_rate}, workers: {num_workers}")


In [ ]:
# Create defult Swin embedding for all baselines
def create_patch_embed(exp_config):
    patch_embed_class = exp_config["patch_embed_class"]
    if patch_embed_class is None:
        return PatchEmbed(patch_size=PATCH_SIZE, in_chans=3, embed_dim=FEATURE_SIZE, norm_layer=nn.LayerNorm, spatial_dims=2)
# For all MSPEs
    return patch_embed_class(
        patch_size=PATCH_SIZE,
        in_chans=3,
        embed_dim=FEATURE_SIZE,
        norm_layer=nn.LayerNorm,
        spatial_dims=2,
        K=DEFAULT_K,
        resolutions=ADE_RESOLUTIONS,
    )

# Create SwinUntetr
def create_model_for_experiment(exp_config, device):
    model = monai.networks.nets.SwinUNETR(
        in_channels=3,
        out_channels=ADE20K_NUM_CLASSES,
        spatial_dims=2,
        patch_size=PATCH_SIZE,
        feature_size=FEATURE_SIZE,
        depths=SWIN_DEPTHS,
        num_heads=SWIN_NUM_HEADS,
        use_v2=exp_config["use_v2"],
        dropout_path_rate=DROPOUT_PATH_RATE, 
    )

   # Fix for patch size 2 -> 4
    model.decoder1 = UnetrUpBlock(
        spatial_dims=2,
        in_channels=FEATURE_SIZE,
        out_channels=FEATURE_SIZE,
        kernel_size=3,
        upsample_kernel_size=PATCH_SIZE,
        norm_name="instance",
        res_block=True,
    )

    # Monai  divisibility by patch_size ** 5
    def _check_input_size(spatial_shape):
        if any(s % STRIDE_DIVISOR for s in spatial_shape):
            raise ValueError(
                f"spatial shape {tuple(spatial_shape)} must be divisible by {STRIDE_DIVISOR}"
            )
    model._check_input_size = _check_input_size

    model.swinViT.patch_embed = create_patch_embed(exp_config)
    model = model.to(device)
    print(f"\nCreated {exp_config['name']}")
    print(f"use_v2={exp_config['use_v2']}, train_mode={exp_config['train_mode']}, patch_size={PATCH_SIZE}")
    print(model.swinViT.patch_embed)
    return model


def get_aspect_preserving_target_size(inputs, effective_resolution, divisor=STRIDE_DIVISOR):
    spatial_shape = inputs.shape[2:]
    spatial_dims = len(spatial_shape)
    current_eff = float(np.prod(spatial_shape)) ** (1.0 / spatial_dims)
    scale = effective_resolution / current_eff

    target_size = []
    for dim in spatial_shape:
        resized_dim = int(round(dim * scale))
        resized_dim = max(divisor, int(round(resized_dim / divisor)) * divisor)
        target_size.append(resized_dim)
    return target_size

# Batch resize
def resize_batch(inputs, labels, effective_resolution):
    if effective_resolution is None:
        return inputs, labels

    spatial_dims = inputs.ndim - 2
    target_size = get_aspect_preserving_target_size(inputs, effective_resolution)
    image_mode = "bilinear" if spatial_dims == 2 else "trilinear"

    inputs = F.interpolate(inputs, size=target_size, mode=image_mode, align_corners=False)
    labels = F.interpolate(labels.float(), size=target_size, mode="nearest").to(labels.dtype)
    return inputs, labels

# Train batch resize
def resize_train_batch(inputs, labels, effective_resolution=None):
    if effective_resolution is None:
        effective_resolution = int(np.random.choice(TRAIN_RESOLUTIONS))
    inputs, labels = resize_batch(inputs, labels, effective_resolution)
    return inputs, labels, int(effective_resolution), tuple(inputs.shape[2:])

# Set WANDB_API_KEY in your environment (or Colab secrets) before running
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

# WB login
def login_wandb():
    if colab:
        wandb.login(key=WANDB_API_KEY)


In [ ]:
# Swin-T ImageNet-1K warm-start utilities

def _extract_state_dict(raw):
    """Unwrap common checkpoint containers to the bare {name: tensor} state dict."""
    if isinstance(raw, dict):
        for k in ("state_dict", "model", "model_state_dict"):
            if isinstance(raw.get(k), dict):
                return raw[k]
    return raw


def get_timm_swin_sd(name):
    """Download (HF-cached) a timm Swin model and return its ImageNet state dict."""
    return timm.create_model(name, pretrained=True).state_dict()


def timm_swin_to_monai(sd):
    """Rename timm Swin keys to MONAI swinViT keys."""
    out = {}
    for k, v in sd.items():
        if k.startswith("head") or k in ("norm.weight", "norm.bias"):
            continue
        k = k.replace("mlp.fc1", "mlp.linear1").replace("mlp.fc2", "mlp.linear2")
        md = re.match(r"^layers\.(\d+)\.downsample\.(.*)$", k)
        if md:
            out[f"layers{int(md.group(1))}.0.downsample.{md.group(2)}"] = v
            continue
        nk = re.sub(r"^layers\.(\d+)\.", lambda m: f"layers{int(m.group(1)) + 1}.0.", k)
        out[nk] = v
    return out


def inspect_checkpoint(source):
    """Print structure."""
    if isinstance(source, dict):
        sd, name = _extract_state_dict(source), "<state_dict>"
    else:
        sd = _extract_state_dict(torch.load(source, map_location="cpu", weights_only=False))
        name = os.path.basename(source)
    print(f"{name}: {len(sd)} tensors")
    pe = next((v for k, v in sd.items() if k.endswith("patch_embed.proj.weight")), None)
    if pe is not None:
        print(f"  patch_embed.proj -> embed_dim={pe.shape[0]}, in_chans={pe.shape[1]}, kernel={tuple(pe.shape[2:])}")
        print(f"  => match FEATURE_SIZE = {pe.shape[0]}")
    for k in list(sd)[:3]:
        print(f"  e.g. {k}  {tuple(sd[k].shape)}")


def load_pretrained_backbone(model, source, seed_stem=True, verbose=True):
    """Warm-start model.swinViT from a pretrained Swin checkpoint. """
    if isinstance(source, dict):
        sd = _extract_state_dict(source)
    else:
        sd = _extract_state_dict(torch.load(source, map_location="cpu", weights_only=False))

    target = model.swinViT.state_dict()
    remap, used = {}, set()
    for tk, tv in target.items():
        for ck, cv in sd.items():
            if ck not in used and ck.endswith(tk) and cv.shape == tv.shape:
                remap[tk] = cv
                used.add(ck)
                break
    model.swinViT.load_state_dict(remap, strict=False)

    if verbose:
        n_stage = sum(1 for k in remap if k.startswith("layers"))
        n_stage_tot = sum(1 for k in target if k.startswith("layers"))
        print(f"swinViT warm start: loaded {len(remap)}/{len(target)} tensors "
              f"(transformer stages {n_stage}/{n_stage_tot})")
        if not remap:
            print("  WARNING: 0 tensors matched -- check FEATURE_SIZE/depths vs checkpoint "
                  "(run inspect_checkpoint).")
        elif len(remap) < len(target):
            left = Counter(k.split(".")[0] for k in target if k not in remap)
            print(f"  left at init by top-level module: {dict(left)} "
                  f"(expected: relative_position_index buffers and layers4 downsample)")

    if seed_stem:
        w = next((v for k, v in sd.items() if k.endswith("patch_embed.proj.weight") and v.ndim in (4, 5)), None)
        b = next((v for k, v in sd.items() if k.endswith("patch_embed.proj.bias")), None)
        pe = model.swinViT.patch_embed
        if w is None:
            if verbose:
                print("  no patch_embed.proj.weight in checkpoint; stem left at init")
        elif hasattr(pe, "patch_kernels"):        # MSPE
            n = 0
            for kern in pe.patch_kernels:
                tgt = kern.weight.data
                if w.shape[1] != tgt.shape[1]:
                    continue
                src = pi_resize(w, tuple(tgt.shape[2:])) if tuple(w.shape[2:]) != tuple(tgt.shape[2:]) else w
                if src.shape == tgt.shape:
                    kern.weight.data.copy_(src.to(tgt))
                    if b is not None and kern.bias is not None:
                        kern.bias.data.copy_(b.to(kern.bias))
                    n += 1
            if verbose:
                print(f"  seeded {n}/{len(pe.patch_kernels)} MSPE kernels from patch-embed (pi_resize)")
        elif hasattr(pe, "proj"):                 # plain PatchEmbed stem
            tgt = pe.proj.weight.data
            if w.shape[1] == tgt.shape[1]:
                src = pi_resize(w, tuple(tgt.shape[2:])) if tuple(w.shape[2:]) != tuple(tgt.shape[2:]) else w
                if src.shape == tgt.shape:
                    pe.proj.weight.data.copy_(src.to(tgt))
                    if b is not None and pe.proj.bias is not None:
                        pe.proj.bias.data.copy_(b.to(pe.proj.bias))
                    if verbose:
                        print("  seeded plain patch-embed proj from checkpoint (pi_resize)")
    return model


def warm_start_swin_imagenet(model, exp_config, verbose=True):
    """Load Swin-T IN-1K weights into model.swinViT."""
    sd = timm_swin_to_monai(get_timm_swin_sd(IMAGENET_SWIN_MODEL))
    return load_pretrained_backbone(model, sd, seed_stem=True, verbose=verbose)


In [ ]:
# Confusion-matrix metrics (aAcc/mAcc/mIoU/macro-F1)

# Accumulate a full confusion matrix 
def update_confmat(cm, logits, labels):
    preds = logits.argmax(dim=1).reshape(-1) # (B*H*W,)
    gts = labels[:, 0].reshape(-1).long()
    valid = gts != IGNORE_INDEX                           
    idx = gts[valid] * ADE20K_NUM_CLASSES + preds[valid]
    binc = torch.bincount(idx, minlength=ADE20K_NUM_CLASSES ** 2)
    return cm + binc.reshape(ADE20K_NUM_CLASSES, ADE20K_NUM_CLASSES)

def metrics_from_confmat(cm):
    cm = cm.double()
    tp = torch.diag(cm)
    gt = cm.sum(dim=1)  # row totals 
    pred = cm.sum(dim=0)                                 
    fn = gt - tp
    fp = pred - tp
    sel = slice(1, ADE20K_NUM_CLASSES)                   
    iou = (tp / (tp + fp + fn))[sel]                      
    acc = (tp / gt)[sel]                                  
    f1 = (2 * tp / (2 * tp + fp + fn))[sel]          
    total = gt[sel].sum()
    return {
        "mean_iou": iou.nanmean().item(),
        "mean_acc": acc.nanmean().item(),
        "mean_f1": f1.nanmean().item(),
        "aacc": (tp[sel].sum() / total).item() if total > 0 else float("nan"),
        "iou_vals": iou.cpu(),
        "acc_vals": acc.cpu(),
        "f1_vals": f1.cpu(),
    }

# Print a compact summary
def print_metric_table(title, result, worst_k=5):
    print(title)
    print(f"  aAcc: {result['aacc']:.4f} | mAcc: {result['mean_acc']:.4f} | "
          f"mIoU: {result['mean_iou']:.4f} | macro-F1: {result['mean_f1']:.4f}")
    print()
    return result

# Log metrics to wandb (summary + per-class series)
def log_eval_result(prefix, result, best_metric_epoch):
    if wandb.run is None:
        return

    log_data = {
        f"{prefix}/mIoU": result["mean_iou"],
        f"{prefix}/mAcc": result["mean_acc"],
        f"{prefix}/macroF1": result["mean_f1"],
        f"{prefix}/aAcc": result["aacc"],
        "best_metric_epoch": best_metric_epoch,
    }
    for c, (iou_v, acc_v, f1_v) in enumerate(
        zip(result["iou_vals"], result["acc_vals"], result["f1_vals"]), start=1
    ):
        log_data[f"{prefix}/class_{c}_iou"] = iou_v.item()
        log_data[f"{prefix}/class_{c}_acc"] = acc_v.item()
        log_data[f"{prefix}/class_{c}_f1"] = f1_v.item()
    wandb.log(log_data)

# Evals robustness only, dont report in the paper
def evaluate_validation_loader(model, data_loader, resolution=None, prefix="validation_native", best_metric_epoch=-1):
    cm = torch.zeros(ADE20K_NUM_CLASSES, ADE20K_NUM_CLASSES, dtype=torch.long, device=device)

    with torch.no_grad():
        for val_data in data_loader:
            val_inputs, val_labels = (
                val_data["image"].to(device, non_blocking=True),
                val_data["label"].to(device, non_blocking=True),
            )
            val_inputs, val_labels = resize_batch(val_inputs, val_labels, resolution)

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                val_outputs = model(val_inputs)

            cm = update_confmat(cm, val_outputs, val_labels)
            del val_outputs, val_inputs, val_labels

    result = metrics_from_confmat(cm)
    title = "\nNative validation metrics" if resolution is None else f"Resized validation metrics at effective res {resolution}"
    print_metric_table(title, result)
    log_eval_result(prefix, result, best_metric_epoch)
    return result

# Real eval
def evaluate_native_sliding_window(model, fullres_loader, roi_size=(RESIZE_LONGEST, RESIZE_LONGEST),
                                   sw_batch_size=4, overlap=0.5,
                                   prefix="validation_native_sw", best_metric_epoch=-1):
    cm = torch.zeros(ADE20K_NUM_CLASSES, ADE20K_NUM_CLASSES, dtype=torch.long, device=device)

    def _predict(patch):
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            return model(patch)

    with torch.no_grad():
        for val_data in fullres_loader:
            val_inputs = val_data["image"].to(device, non_blocking=True)
            val_labels = val_data["label"].to(device, non_blocking=True)
            val_outputs = sliding_window_inference(
                val_inputs, roi_size=roi_size, sw_batch_size=sw_batch_size,
                predictor=_predict, overlap=overlap, mode="gaussian",
            )
            cm = update_confmat(cm, val_outputs, val_labels)
            del val_outputs, val_inputs, val_labels

    result = metrics_from_confmat(cm)
    print_metric_table("\nNative validation metrics (sliding window)", result)
    log_eval_result(prefix, result, best_metric_epoch)
    return result

# Compute val evals
def evaluate_validation_all_resolutions(model, val_loader, val_loader_fullres, exp_name, best_metric_epoch,
                                        checkpoint_path, roi_size=(RESIZE_LONGEST, RESIZE_LONGEST), sw_overlap=0.5):
    print(f"\n{'=' * 40}")
    print(f"VALIDATION EVALUATION: {exp_name}")
    print(f"{'=' * 40}")

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    model.eval()

    native_result = evaluate_native_sliding_window(
        model, val_loader_fullres, roi_size=roi_size, overlap=sw_overlap,
        prefix="validation_native_sw", best_metric_epoch=best_metric_epoch,
    )
    torch.cuda.empty_cache()  # release peak allocation
    gc.collect()

    # Downsampled effective resolutions: keep the single-pass model(x) evaluation.
    multi_resolution_results = {}
    for hw in TEST_RESOLUTIONS:
        multi_resolution_results[hw] = evaluate_validation_loader(
            model, val_loader, resolution=hw,
            prefix=f"validation_res_{hw}", best_metric_epoch=best_metric_epoch,
        )

    if wandb.run is not None:
        wandb.log({
            "validation/mIoU": native_result["mean_iou"],
            "validation/mAcc": native_result["mean_acc"],
            "validation/macroF1": native_result["mean_f1"],
            "validation/aAcc": native_result["aacc"],
            "best_metric_epoch": best_metric_epoch,
        })

    return {"native": native_result, "multi_res": multi_resolution_results}

In [ ]:
# See kernels drift
def analyze_kernel_drift(initial_state, model, exp_name):
    patch_embed = model.swinViT.patch_embed
    if not hasattr(patch_embed, "patch_kernels"):
        print(f"{exp_name}: no MSPE kernels; skipping drift analysis.")
        return None

    final_state = patch_embed.state_dict()
    metrics_l2 = []
    metrics_cos = []
    metrics_max = []

    print(f"\nKernel drift analysis: {exp_name}")
    print("Kernel | L2 Dist  | Cosine Sim | Max Diff")
    print("-" * 44)

    for k in range(len(patch_embed.patch_kernels)):
        weight_key = f"patch_kernels.{k}.weight"
        if weight_key not in initial_state or weight_key not in final_state:
            raise KeyError(f"Missing MSPE weight key for drift analysis: {weight_key}")

        w_init = initial_state[weight_key].detach().flatten().cpu()
        w_final = final_state[weight_key].detach().flatten().cpu()
        l2_dist = torch.norm(w_final - w_init, p=2).item()
        cos_sim = F.cosine_similarity(w_final.unsqueeze(0), w_init.unsqueeze(0)).item()
        max_diff = torch.max(torch.abs(w_final - w_init)).item()

        metrics_l2.append(l2_dist)
        metrics_cos.append(cos_sim)
        metrics_max.append(max_diff)
        print(f"{k:>6} | {l2_dist:>8.4f} | {cos_sim:>10.4f} | {max_diff:>8.4f}")

    drift = {
        "l2": metrics_l2,
        "cosine": metrics_cos,
        "max_diff": metrics_max,
    }

    if wandb.run is not None:
        log_data = {}
        for k, (l2_dist, cos_sim, max_diff) in enumerate(zip(metrics_l2, metrics_cos, metrics_max)):
            log_data[f"kernel_drift/kernel_{k}_l2"] = l2_dist
            log_data[f"kernel_drift/kernel_{k}_cosine"] = cos_sim
            log_data[f"kernel_drift/kernel_{k}_max_diff"] = max_diff
        wandb.log(log_data)

    x = np.arange(len(metrics_l2))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(x, metrics_l2)
    axes[0].set_title("L2 distance")
    axes[0].set_xlabel("Kernel")
    axes[1].bar(x, metrics_cos)
    axes[1].set_title("Cosine similarity")
    axes[1].set_xlabel("Kernel")
    axes[2].bar(x, metrics_max)
    axes[2].set_title("Max absolute diff")
    axes[2].set_xlabel("Kernel")
    fig.suptitle(f"MSPE kernel drift: {exp_name}")
    plt.tight_layout()

    if wandb.run is not None:
        wandb.log({"kernel_drift/plot": wandb.Image(fig)})
    plt.show()

    return drift

In [ ]:
# Train helpers

# Move to device and resize batch
def _prepare_training_batch(batch_data, exp_config):
    inputs, labels = (
        batch_data["image"].to(device, non_blocking=True),
        batch_data["label"].to(device, non_blocking=True),
    )

    if exp_config["train_mode"] == "resize_aug":
        inputs, labels, _, _ = resize_train_batch(inputs, labels)

    return inputs, labels

# Notebook local MSPE train step adapted for patch_size = 4
def mspe_swin_train_step(model, img, label, loss_fn, lam=1.0):
    patch_embed = model.swinViT.patch_embed
    hw_list = patch_embed.sample_resolutions()

    total_loss = 0.0

    # K resolution specific forwards
    for k, eff_target_k in enumerate(hw_list):
        img_k, label_k = resize_batch(img, label, eff_target_k) # local
        logits_k = mspe_swin_forward(model, img_k, func_idx=k)
        total_loss = total_loss + loss_fn(logits_k, label_k)

    # Original res forward
    logits_orig = model(img)
    total_loss = total_loss + lam * loss_fn(logits_orig, label)
    # Normalize loss for vanilla train comp
    return total_loss / (K + 1)

# Compute loss
def _compute_training_loss(model, inputs, labels, loss_function, exp_config):
    if exp_config["train_mode"] == "mspe":
        # NOTE: MSPE train step performs multi-res routing
        return mspe_swin_train_step(model, inputs, labels, loss_function, lam=1.0)

    outputs = model(inputs)
    return loss_function(outputs, labels)

# No weight decay for pos bias
def _is_no_decay(name, param):
    return param.ndim <= 1 or name.endswith("relative_position_bias_table")

# Differential LR
def build_optimizer(model):
    if SWIN_INIT is None:
        return torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    groups = {"backbone_decay": [], "backbone_no_decay": [], "head_decay": [], "head_no_decay": []}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        # patch_embed goes with  the backbone
        prefix = "backbone" if name.startswith("swinViT.") else  "head"
        suffix = "no_decay" if _is_no_decay(name, p) else "decay"
        groups[f"{prefix}_{suffix}"].append(p)

    return torch.optim.AdamW(
        [
            {"params": groups["backbone_decay"], "lr": BACKBONE_LR, "weight_decay": WD},
            {"params": groups["backbone_no_decay"], "lr": BACKBONE_LR, "weight_decay": 0.0},
            {"params": groups["head_decay"], "lr": LR, "weight_decay": WD},
            {"params": groups["head_no_decay"], "lr": LR, "weight_decay": 0.0},
        ],
    )

# Training loop
def train_variant(model, train_loader, val_loader, exp_config):
    exp_name = exp_config["name"]
    checkpoint_path = exp_config["checkpoint_path"]
    last_state_path = checkpoint_path.replace("best_metric_", "last_state_")

    ce_loss = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    def loss_function(logits, targets):
        return ce_loss(logits, targets[:, 0].long())

    optimizer = build_optimizer(model)

    # Warm start -> short linear warmup
    if SWIN_INIT is not None:
        warmup_scheduler = LinearLR(optimizer, start_factor=1/20, total_iters=WARMUP_EPOCHS)
        cosine_scheduler = CosineAnnealingLR(optimizer, T_max=max(1, MAX_EPOCHS - WARMUP_EPOCHS), eta_min=1e-6)
        scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS])
    else:
        scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)

    scaler = GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))

    best_metric = -1  # best validation mIoU
    best_metric_epoch = -1
    epoch_loss_values = []
    metric_values = []
    epochs_no_improve = 0
    completed_epochs = 0
    start_epoch = 0

    # R2 resume insurance
    if RESUME and os.path.exists(last_state_path):
        state = torch.load(last_state_path, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        start_epoch = state["epoch"]
        best_metric = state["best_metric"]
        best_metric_epoch = state["best_metric_epoch"]
        epochs_no_improve = state["epochs_no_improve"]
        epoch_loss_values = state["epoch_loss_values"]
        metric_values = state["metric_values"]
        completed_epochs = start_epoch
        print(f"Resumed training state from {last_state_path} at epoch {start_epoch}")

    total_start = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS):
        epoch_start = time.time()
        print("-" * 10)
        print(f"{exp_name}: epoch {epoch + 1}/{MAX_EPOCHS}")
        model.train()
        epoch_loss = 0
        step = 0

        for batch_data in train_loader:
            step += 1
            inputs, labels = _prepare_training_batch(batch_data, exp_config)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                loss = _compute_training_loss(model, inputs, labels, loss_function, exp_config)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

            log_data = {
                "train/step_loss": loss.item(),
                "train/amp_scale": scaler.get_scale(),
                "train/grad_norm": grad_norm.item(),
            }
            if wandb.run is not None:
                wandb.log(log_data)

            if step % 10 == 0 or step == len(train_loader):  # clean stdout
                print(f"{step}/{len(train_loader)}, train_loss: {loss.item():.4f}")

        epoch_loss /= step
        epoch_loss_values.append(epoch_loss)
        completed_epochs = epoch + 1
        epoch_time_sec = time.time() - epoch_start
        head_lr = optimizer.param_groups[-1]["lr"]
        backbone_lr = optimizer.param_groups[0]["lr"]

        print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")
        print(f"time per epoch: {epoch_time_sec:.2f} s")

        if wandb.run is not None:
            wandb.log({
                "train/epoch_loss": epoch_loss,
                "lr": head_lr,
                "lr_backbone": backbone_lr,
                "epoch": epoch + 1,
                "train/time_per_epoch": epoch_time_sec,
            })

        if (epoch + 1) % VAL_INTERVAL == 0:
            model.eval()
            with torch.no_grad():
                cm = torch.zeros(ADE20K_NUM_CLASSES, ADE20K_NUM_CLASSES, dtype=torch.long, device=device)
                for val_data in val_loader:
                    val_inputs, val_labels = (
                        val_data["image"].to(device, non_blocking=True),
                        val_data["label"].to(device, non_blocking=True),
                    )
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                        val_outputs = model(val_inputs)
                    cm = update_confmat(cm, val_outputs, val_labels)
                    del val_outputs, val_inputs, val_labels

                # Select on mIoU
                val_metrics = metrics_from_confmat(cm)
                metric = val_metrics["mean_iou"]
                dice = val_metrics["mean_f1"]  # macro F1 == mean  Dice
                metric_values.append(metric)

                if metric > best_metric:
                    best_metric = metric
                    best_metric_epoch = epoch + 1
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), checkpoint_path)
                    print(f"saved new best metric model to {checkpoint_path}")
                    print(
                        f"current epoch: {epoch + 1} current mIoU: {metric:.3f} current Dice: {dice:.3f}\n"
                        f"best validation mIoU: {best_metric:.3f} at epoch: {best_metric_epoch}"
                    )
                else:
                    epochs_no_improve += VAL_INTERVAL

                if wandb.run is not None:
                    wandb.log({"val mIoU": metric, "val Dice": dice, "epoch": epoch + 1})

                if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
                    print(f"Early stopping triggered at epoch {epoch + 1}. No improvement for {EARLY_STOPPING_PATIENCE} epochs.")
                    if wandb.run is not None:
                        wandb.log({"early_stop_epoch": epoch + 1})
                    break

        scheduler.step()

        # Resume safety every 10 epochs 
        if (epoch + 1) % 10 == 0:
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "epoch": epoch + 1,
                "best_metric": best_metric,
                "best_metric_epoch": best_metric_epoch,
                "epochs_no_improve": epochs_no_improve,
                "epoch_loss_values": epoch_loss_values,
                "metric_values": metric_values,
            }, last_state_path)

        gc.collect()

    total_time = time.time() - total_start
    avg_epoch_time = total_time / max(completed_epochs, 1)
    print(f"total epochs: {completed_epochs}, total training time: {total_time / 60:.2f} min, average time per epoch: {avg_epoch_time:.2f} s")
    print(f"train completed, best validation mIoU: {best_metric:.4f} at epoch: {best_metric_epoch}")

    return best_metric, best_metric_epoch, epoch_loss_values, metric_values

In [ ]:
# Experimenal loop

login_wandb()
all_validation_results = {}
all_training_curves = {}
all_drift = {}

for exp_idx, exp in enumerate(EXPERIMENTS):
    set_determinism(seed=2026)
    torch.backends.cudnn.benchmark = True
    torch.use_deterministic_algorithms(False)

    exp_name = exp["name"]
    run_name = exp["run_name"]
    print(f"\n{'=' * 60}")
    print(f"EXPERIMENT {exp_idx + 1}/{len(EXPERIMENTS)}: {run_name}")
    print(exp["description"])
    print(f"{'=' * 60}\n")

    # W&B interrupted execution  
    if wandb.run is not None:
        wandb.finish()

    wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        save_code=True,
        group=exp_name,
        tags=[DATASET_TAG, RECIPE_TAG],
        mode=None if colab else "disabled",   
        config={
            "recipe": RECIPE_TAG,
            "max_epochs": MAX_EPOCHS,
            "val_interval": VAL_INTERVAL,
            "feature_size": FEATURE_SIZE,
            "patch_size": PATCH_SIZE,
            "depths": SWIN_DEPTHS,
            "num_heads": SWIN_NUM_HEADS,
            "gradient_checkpointing": False,
            "dropout_path_rate": DROPOUT_PATH_RATE,
            "learning_rate": LR,
            "backbone_lr": BACKBONE_LR if SWIN_INIT is not None else LR,
            "warmup_epochs": WARMUP_EPOCHS if SWIN_INIT is not None else 0,
            "weight_decay": WD,
            "no_decay_on": ["norm", "bias", "relative_position_bias_table"],
            "grad_clip": GRAD_CLIP,
            "optimizer": "AdamW",
            "batch_size": batch_size,
            "base_longest": BASE_LONGEST,
            "crop_size": CROP_SIZE,
            "scale_jitter": list(SCALE_JITTER),
            "hflip_prob": 0.5,
            "resize_longest": RESIZE_LONGEST,
            "num_classes": ADE20K_NUM_CLASSES,
            "loss_type": "CrossEntropy",
            "metrics": ["aAcc", "mAcc", "mIoU", "macroF1"],
            "selection_metric": "mIoU",
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "dataset": DATASET_TAG,
            "train_mode": exp["train_mode"],
            "use_v2": exp["use_v2"],
            "swin_init": SWIN_INIT if PRETRAINED_CKPT is None else f"full:{os.path.basename(PRETRAINED_CKPT)}",
            "imagenet_model": IMAGENET_SWIN_MODEL if SWIN_INIT == "imagenet" else None,
            "K": DEFAULT_K if exp["patch_embed_class"] else 1,
            "train_resolutions": TRAIN_RESOLUTIONS if exp["train_mode"] in ("resize_aug", "mspe") else None,
            "test_resolutions": TEST_RESOLUTIONS,
            "native_eval": "sliding_window",
            "sw_roi_size": RESIZE_LONGEST,
            "sw_overlap": 0.5,
        },
    )

    try:
        model = create_model_for_experiment(exp, device)
        # Encoder init
        if PRETRAINED_CKPT is not None:
            model.load_state_dict(torch.load(PRETRAINED_CKPT, weights_only=True))
            print(f"Loaded full pretrained weights from {PRETRAINED_CKPT}")
        elif SWIN_INIT == "imagenet":
            warm_start_swin_imagenet(model, exp)
        else:
            print(f"No pretrained weights specified; training {exp_name} from scratch")

        #  MSPE kernel init drift
        initial_mspe_state = None
        if exp["patch_embed_class"] is not None:
            initial_mspe_state = copy.deepcopy(model.swinViT.patch_embed.state_dict())

        best_metric, best_metric_epoch, epoch_losses, metric_vals = train_variant(model, train_loader, val_loader, exp)
        all_training_curves[exp_name] = {
            "epoch_losses": epoch_losses,
            "metric_values": metric_vals,
            "best_metric": best_metric,
            "best_metric_epoch": best_metric_epoch,
            "checkpoint_path": exp["checkpoint_path"],
        }

        validation_results = evaluate_validation_all_resolutions(model, val_loader, val_loader_fullres, exp_name, best_metric_epoch, checkpoint_path=exp["checkpoint_path"])
        all_validation_results[exp_name] = validation_results

        if initial_mspe_state is not None:
            all_drift[exp_name] = analyze_kernel_drift(initial_mspe_state, model, exp_name)
    finally:
        # guarantee the run closes on interrupt
        if wandb.run is not None:
            wandb.finish()

    del model
    torch.cuda.empty_cache()
    print(f"\n--- Completed {exp_name}")

print("\n" + "=" * 60)
print("ALL FULL-TRAINING EXPERIMENTS COMPLETED")
print("=" * 60)


In [ ]:
# Summary tables

print(f"\n{'=' * 80}")
print("VALIDATION RESULTS SUMMARY: Full training")
print(f"{'=' * 80}\n")

res_cols = ["Native(SW)"] + [str(r) for r in TEST_RESOLUTIONS]
header = f"{'Condition':<26} | " + " | ".join(f"{c:>10}" for c in res_cols) + " |"
sep = "-" * len(header)

# One table per metric across native + test resolutions 
metric_specs = [
    ("mIoU", "mean_iou"),
    ("mAcc", "mean_acc"),
    ("macro-F1", "mean_f1"),
    ("aAcc", "aacc"),
]
for label, key in metric_specs:
    print(f"\nValidation {label} (classes 1-150):")
    print(sep)
    print(header)
    print(sep)
    for exp_name, results in all_validation_results.items():
        row = f"{exp_name:<26} | {results['native'][key]:>10.4f}"
        for hw in TEST_RESOLUTIONS:
            row += f" | {results['multi_res'][hw][key]:>10.4f}"
        row += " |"
        print(row)
    print(sep)

print("\nTraining Summary:")
print(f"{'Condition':<26} | {'Best mIoU':>10} | {'Best Epoch':>10} | {'Final Loss':>10}")
print("-" * 68)
for exp_name, curves in all_training_curves.items():
    final_loss = curves["epoch_losses"][-1] if curves["epoch_losses"] else float("nan")
    print(f"{exp_name:<26} | {curves['best_metric']:>10.4f} | {curves['best_metric_epoch']:>10d} | {final_loss:>10.4f}")

if all_drift:
    print("\nMSPE Kernel Drift Summary:")
    drift_header = f"{'Condition':<26} | {'Kernel':>6} | {'L2':>8} | {'Cosine':>10} | {'Max Diff':>8}"
    print(drift_header)
    print("-" * len(drift_header))
    for exp_name, drift in all_drift.items():
        for k, (l2_dist, cos_sim, max_diff) in enumerate(zip(drift["l2"], drift["cosine"], drift["max_diff"])):
            print(f"{exp_name:<26} | {k:>6d} | {l2_dist:>8.4f} | {cos_sim:>10.4f} | {max_diff:>8.4f}")

In [ ]:
# Training curves

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for exp_name, curves in all_training_curves.items():
    epochs = list(range(1, len(curves["epoch_losses"]) + 1))
    ax.plot(epochs, curves["epoch_losses"], label=exp_name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training Loss")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
for exp_name, curves in all_training_curves.items():
    val_epochs = [VAL_INTERVAL * (i + 1) for i in range(len(curves["metric_values"]))]
    ax.plot(val_epochs, curves["metric_values"], label=exp_name, marker="o", markersize=3)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation mIoU")
ax.set_title("Validation mIoU")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Quick sanity check for trained checkpoints

def to_display_image(tensor_chw):
    image = tensor_chw.detach().cpu()
    if image.shape[0] in (3, 4):
        return _denormalize_for_display(image[:3]).permute(1, 2, 0)
    return image[0]


vis_resolutions = [None] + list(TEST_RESOLUTIONS)

login_wandb()

os.makedirs(EVAL_DIR, exist_ok=True)

for exp in EXPERIMENTS:
    ckpt = exp["checkpoint_path"]
    print(f"Visualizing {exp['name']} from {ckpt}")

    model_viz = create_model_for_experiment(exp, device)
    model_viz.load_state_dict(torch.load(ckpt, weights_only=True))
    model_viz.eval()

    with torch.no_grad():
        val_data = first(val_loader)
        val_inputs = val_data["image"].to(device, non_blocking=True)
        val_labels = val_data["label"].to(device, non_blocking=True)

        n_cols = len(vis_resolutions)
        fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 12))
        if n_cols == 1:
            axes = axes.reshape(3, 1)

        for col, effective_resolution in enumerate(vis_resolutions):
            inputs_r, labels_r = resize_batch(val_inputs, val_labels, effective_resolution)

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                outputs = model_viz(inputs_r)

            image = to_display_image(inputs_r[0])
            label = labels_r[0, 0].detach().cpu()
            pred = torch.argmax(outputs, dim=1).detach().cpu()[0]

            h, w = inputs_r.shape[-2], inputs_r.shape[-1]
            col_title = f"native\n{h}x{w}" if effective_resolution is None else f"eff {effective_resolution}\n{h}x{w}"

            axes[0, col].imshow(image)
            axes[0, col].set_title(col_title)
            axes[0, col].axis("off")
            axes[1, col].imshow(label, cmap="viridis")
            axes[1, col].set_title("label")
            axes[1, col].axis("off")
            axes[2, col].imshow(pred, cmap="viridis")
            axes[2, col].set_title("prediction")
            axes[2, col].axis("off")

        fig.suptitle(exp["name"], fontsize=14)
        plt.tight_layout()

        viz_path = os.path.join(EVAL_DIR, f"viz_{exp['run_name']}.png")
        fig.savefig(viz_path, dpi=300, bbox_inches="tight")
        print(f"saved visualization to {viz_path}")

        # Save inference image to wandb
        wandb.init(
            project=WANDB_PROJECT,
            name=f"{exp['run_name']}_viz",
            group=exp["name"],
            tags=[DATASET_TAG, RECIPE_TAG, "viz"],
            mode=None if colab else "disabled",
        )
        wandb.log({"predictions": wandb.Image(fig)})
        wandb.finish()

        plt.show()
        plt.close(fig)

    del model_viz
    torch.cuda.empty_cache()

In [ ]:
# Save requirements
# !pip freeze > /content/drive/MyDrive/MSPE/requirements_frozen.txt

In [ ]:
# Disconnect from runtime 
runtime.unassign()